# Machine Learning Methylation Experiment
___
Here, we'll narrow down PPMI project 120's m-values to the 450k range of CpG's and train an XGBoost classifier. Afterwards, we'll test the model against GSE111629.

In [1]:
import pandas as pd
src_data_path = "/root/data/ppmi/PPMI_data.parquet"
df_ppmi = pd.read_parquet(src_data_path)
df_ppmi = df_ppmi[df_ppmi["Sample_Group"].isin(["Control", "PD"])]

FileNotFoundError: [Errno 2] No such file or directory: '/root/data/ppmi/PPMI_data.parquet'

In [ ]:
df_beta_means = pd.read_csv("/root/data/ppmi/beta_means.csv", index_col=0)

In [ ]:
df_sign_beta = df_beta_means[df_beta_means["delta_beta"].abs() > 0.02].sort_values("delta_beta", ascending=False).head(10)

In [3]:
%conda install conda-forge::optuna

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /root/miniconda3/envs/rapids-26.04

  added / updated specs:
    - conda-forge::optuna


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    alembic-1.18.4             |  py313h06a4308_0         461 KB
    colorlog-6.10.1            |  py313h06a4308_0          22 KB
    greenlet-3.3.2             |  py313h7354ed3_0         238 KB
    mako-1.3.11                |  py313h06a4308_0         229 KB
    optuna-3.6.1               |     pyhd8ed1ab_0         218 KB  conda-forge
    sqlalchemy-2.0.48          |  py313h357f0b5_0         6.0 MB
    ------------------------------------------------------------
                                           Total:         7.2 MB

The following NEW packages will be INSTALLED:

  a

In [4]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from cuml.preprocessing import StandardScaler
import optuna
from sklearn.model_selection import StratifiedKFold, train_test_split
from cuml.linear_model import LogisticRegression
import xgboost as xgb
import gc
from sklearn.metrics import average_precision_score, roc_auc_score

def cv_pipeline(X_all, Y_all):
    outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_aucs_roc = []
    fold_aucs_pr = []
    variances = np.var(X_all, axis=0)
    threshold = np.percentile(variances, 75) 
    high_var_mask = variances > threshold
    X_high_var = X_all[:, high_var_mask]
    print(f"Reduced features from {X_all.shape[1]} to {X_high_var.shape[1]} based on variance.")

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_high_var, Y_all)):
        print(f"\n=================== FOLD {fold + 1} ===================")
        
        X_train, Y_train = X_high_var[train_idx], Y_all[train_idx]
        X_test, Y_test = X_high_var[test_idx], Y_all[test_idx]
        
        print("Feature Selection...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        selector = SelectKBest(score_func=f_classif, k=500)
        if hasattr(X_train_scaled, 'get'): 
            X_train_cpu = X_train_scaled.get()
            Y_train_cpu = Y_train.get()
            X_test_cpu = X_test_scaled.get()
        else:
            X_train_cpu = X_train_scaled
            Y_train_cpu = Y_train
            X_test_cpu = X_test_scaled

        selector.fit(X_train_cpu, Y_train_cpu)

        X_train_sel = selector.transform(X_train_cpu)
        X_test_sel  = selector.transform(X_test_cpu)
        
        model = xgb.XGBClassifier(
            tree_method="hist", 
            device="cuda",
            objective="binary:logistic",
            n_estimators=200,
            max_depth=2,
            learning_rate=0.05,
            colsample_bytree=0.3,
            subsample=0.8,
            random_state=42,
            scale_pos_weight=(Y_train_cpu == 0).sum() / (Y_train_cpu == 1).sum()
        )
    
        model.fit(X_train_sel, Y_train_cpu, verbose=False)

        preds = model.predict_proba(X_test_sel)[:, 1]
    
        auc_roc = roc_auc_score(Y_test, preds)
        auc_pr = average_precision_score(Y_test, preds)
        
        fold_aucs_roc.append(auc_roc)
        fold_aucs_pr.append(auc_pr)

        print(f"ROC-AUC: {auc_roc:.4f} | PR-AUC: {auc_pr:.4f}")

        del model, selector
        gc.collect()

    print("\n================ FINAL RESULTS ================")
    print(f"Rigorous Cross-Validated ROC-AUC: {np.mean(fold_aucs_roc):.4f} ± {np.std(fold_aucs_roc):.4f}")
    print(f"Rigorous Cross-Validated PR-AUC: {np.mean(fold_aucs_pr):.4f} ± {np.std(fold_aucs_pr):.4f}")

In [5]:
target_var = "Sample_Group"
meta_cols_ppmi = [col for col in df_ppmi.columns if not (col.startswith("cg") or col.startswith("ch"))]
Y_pd_ppmi = df_ppmi[target_var]
X_pd_ppmi = df_ppmi.drop(columns=meta_cols_ppmi)

Y_pd_ppmi = Y_pd_ppmi.map({"Control": 0, "PD": 1})
X_ppmi = X_pd_ppmi.to_numpy(dtype="float32")
Y_ppmi = Y_pd_ppmi.to_numpy(dtype="float32")

In [6]:
cv_pipeline(X_ppmi, Y_ppmi)

Reduced features from 770664 to 192666 based on variance.

=================== FOLD 1 ===================
Feature Selection...


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/xgboost/core.py:751: UserWarning: [18:04:34] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1774436377359/work/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


ROC-AUC: 0.7619 | PR-AUC: 0.8871

=================== FOLD 2 ===================
Feature Selection...
ROC-AUC: 0.5476 | PR-AUC: 0.8028

=================== FOLD 3 ===================
Feature Selection...
ROC-AUC: 0.7381 | PR-AUC: 0.9084

=================== FOLD 4 ===================
Feature Selection...
ROC-AUC: 0.8651 | PR-AUC: 0.9491

=================== FOLD 5 ===================
Feature Selection...
ROC-AUC: 0.4365 | PR-AUC: 0.6975

=================== FOLD 6 ===================
Feature Selection...
ROC-AUC: 0.5833 | PR-AUC: 0.8493

=================== FOLD 7 ===================
Feature Selection...
ROC-AUC: 0.6387 | PR-AUC: 0.8739

=================== FOLD 8 ===================
Feature Selection...
ROC-AUC: 0.6639 | PR-AUC: 0.7863

=================== FOLD 9 ===================
Feature Selection...
ROC-AUC: 0.5546 | PR-AUC: 0.7747

=================== FOLD 10 ===================
Feature Selection...
ROC-AUC: 0.6891 | PR-AUC: 0.8497

================ FINAL RESULTS ================

In [7]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from cuml.preprocessing import StandardScaler 
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression # <-- THE NEW ALGORITHM
import gc
from sklearn.metrics import average_precision_score, roc_auc_score

def cv_pipeline_linear(X_all, Y_all):
    # Back to 10 folds for maximum validation accuracy
    outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) 
    fold_aucs_roc = []
    fold_aucs_pr = []
    
    # --- 1. Unsupervised Pre-Filtering ---
    variances = np.var(X_all, axis=0)
    threshold = np.percentile(variances, 80) 
    high_var_mask = variances > threshold
    X_high_var = X_all[:, high_var_mask]
    print(f"Reduced features from {X_all.shape[1]} to {X_high_var.shape[1]} based on variance.")

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_high_var, Y_all)):
        print(f"\n=================== FOLD {fold + 1} ===================")
        
        X_train, Y_train = X_high_var[train_idx], Y_all[train_idx]
        X_test, Y_test = X_high_var[test_idx], Y_all[test_idx]
        
        print("Scaling and Feature Selection...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        selector = SelectKBest(score_func=f_classif, k=500)
        
        # Ensure arrays are on CPU for scikit-learn compatibility
        if hasattr(X_train_scaled, 'get'): 
            X_train_cpu = X_train_scaled.get()
            Y_train_cpu = Y_train.get() if hasattr(Y_train, 'get') else Y_train
            X_test_cpu = X_test_scaled.get()
        else:
            X_train_cpu = X_train_scaled
            Y_train_cpu = Y_train
            X_test_cpu = X_test_scaled

        # Fit and transform features
        X_train_sel = selector.fit_transform(X_train_cpu, Y_train_cpu)
        X_test_sel  = selector.transform(X_test_cpu)
        
        # --- 2. THE ALGORITHM SWAP: Penalized Logistic Regression ---
        print("Training Linear Model...")
        model = LogisticRegression(
            penalty='l2',              # Ridge penalty to shrink noisy coefficients
            C=0.1,                     # Strong regularization (lower = stronger)
            class_weight='balanced',   # Automatically handles the 309 vs 122 imbalance
            solver='liblinear',        # Great solver for small datasets
            max_iter=1000,
            random_state=42
        )
        
        model.fit(X_train_sel, Y_train_cpu)

        # --- 3. EVALUATION ---
        preds = model.predict_proba(X_test_sel)[:, 1]
        
        auc_roc = roc_auc_score(Y_test, preds)
        auc_pr = average_precision_score(Y_test, preds)
        
        fold_aucs_roc.append(auc_roc)
        fold_aucs_pr.append(auc_pr)

        print(f"ROC-AUC: {auc_roc:.4f} | PR-AUC: {auc_pr:.4f}")

        del model, selector
        gc.collect()

    print("\n================ FINAL RESULTS ================")
    print(f"Linear Model ROC-AUC: {np.mean(fold_aucs_roc):.4f} ± {np.std(fold_aucs_roc):.4f}")
    print(f"Linear Model PR-AUC: {np.mean(fold_aucs_pr):.4f} ± {np.std(fold_aucs_pr):.4f}")

In [8]:
cv_pipeline_linear(X_ppmi, Y_ppmi)

Reduced features from 770664 to 154133 based on variance.

=================== FOLD 1 ===================
Scaling and Feature Selection...
Training Linear Model...
ROC-AUC: 0.7619 | PR-AUC: 0.9016

=================== FOLD 2 ===================
Scaling and Feature Selection...


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Training Linear Model...
ROC-AUC: 0.6825 | PR-AUC: 0.8790

=================== FOLD 3 ===================
Scaling and Feature Selection...


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Training Linear Model...
ROC-AUC: 0.7222 | PR-AUC: 0.8669

=================== FOLD 4 ===================
Scaling and Feature Selection...


...

/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Training Linear Model...
ROC-AUC: 0.8151 | PR-AUC: 0.9248

=================== FOLD 10 ===================
Scaling and Feature Selection...


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Training Linear Model...
ROC-AUC: 0.6218 | PR-AUC: 0.8203

================ FINAL RESULTS ================
Linear Model ROC-AUC: 0.6784 ± 0.0851
Linear Model PR-AUC: 0.8518 ± 0.0632


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [9]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from cuml.preprocessing import StandardScaler 
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import gc

# Import all the new metrics
from sklearn.metrics import (
    average_precision_score, 
    roc_auc_score, 
    accuracy_score, 
    f1_score, 
    precision_score, 
    recall_score
)

def cv_pipeline_ensemble(X_all, Y_all):
    outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) 
    
    # Buffers for all metrics
    metrics = {
        'roc_auc': [], 'pr_auc': [], 'accuracy': [], 
        'f1': [], 'precision': [], 'recall': []
    }
    
    # --- 1. Unsupervised Pre-Filtering ---
    variances = np.var(X_all, axis=0)
    threshold = np.percentile(variances, 80) 
    high_var_mask = variances > threshold
    X_high_var = X_all[:, high_var_mask]
    
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_high_var, Y_all)):
        print(f"\n=================== FOLD {fold + 1} ===================")
        
        X_train, Y_train = X_high_var[train_idx], Y_all[train_idx]
        X_test, Y_test = X_high_var[test_idx], Y_all[test_idx]
        
        # Scaling
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        # Feature Selection (Top 500)
        selector = SelectKBest(score_func=f_classif, k=500)
        
        if hasattr(X_train_scaled, 'get'): 
            X_train_cpu = X_train_scaled.get()
            Y_train_cpu = Y_train.get() if hasattr(Y_train, 'get') else Y_train
            X_test_cpu = X_test_scaled.get()
        else:
            X_train_cpu = X_train_scaled
            Y_train_cpu = Y_train
            X_test_cpu = X_test_scaled

        X_train_sel = selector.fit_transform(X_train_cpu, Y_train_cpu)
        X_test_sel  = selector.transform(X_test_cpu)
        
        # --- 2. TRAIN BASE MODEL 1: Logistic Regression ---
        logreg = LogisticRegression(
            penalty='l2', C=0.1, class_weight='balanced', 
            solver='liblinear', max_iter=1000, random_state=42
        )
        logreg.fit(X_train_sel, Y_train_cpu)
        preds_logreg = logreg.predict_proba(X_test_sel)[:, 1]

        # --- 3. TRAIN BASE MODEL 2: Constrained XGBoost ---
        xgb_model = xgb.XGBClassifier(
            tree_method="hist", device="cuda", objective="binary:logistic",
            n_estimators=200, max_depth=2, learning_rate=0.05,
            colsample_bytree=0.3, subsample=0.8, random_state=42,
            scale_pos_weight=(Y_train_cpu == 0).sum() / (Y_train_cpu == 1).sum()
        )
        xgb_model.fit(X_train_sel, Y_train_cpu, verbose=False)
        preds_xgb = xgb_model.predict_proba(X_test_sel)[:, 1]

        # --- 4. THE ENSEMBLE ---
        # 1. Soft Voting (Averaging Probabilities)
        final_ensemble_probs = (preds_logreg + preds_xgb) / 2.0
        
        # 2. Hard Class Conversion (Threshold at 0.5)
        final_class_preds = (final_ensemble_probs >= 0.5).astype(int)
        
        # --- 5. EVALUATION ---
        # Probability-based metrics
        metrics['roc_auc'].append(roc_auc_score(Y_test, final_ensemble_probs))
        metrics['pr_auc'].append(average_precision_score(Y_test, final_ensemble_probs))
        
        # Class-based metrics
        metrics['accuracy'].append(accuracy_score(Y_test, final_class_preds))
        metrics['f1'].append(f1_score(Y_test, final_class_preds))
        metrics['precision'].append(precision_score(Y_test, final_class_preds))
        metrics['recall'].append(recall_score(Y_test, final_class_preds))

        print(f"ROC-AUC: {metrics['roc_auc'][-1]:.4f} | PR-AUC: {metrics['pr_auc'][-1]:.4f}")
        print(f"Accuracy: {metrics['accuracy'][-1]:.4f} | F1: {metrics['f1'][-1]:.4f}")

        del logreg, xgb_model, selector
        gc.collect()

    print("\n================ FINAL ENSEMBLE RESULTS ================")
    print(f"Mean ROC-AUC:  {np.mean(metrics['roc_auc']):.4f} ± {np.std(metrics['roc_auc']):.4f}")
    print(f"Mean PR-AUC:   {np.mean(metrics['pr_auc']):.4f} ± {np.std(metrics['pr_auc']):.4f}")
    print(f"Mean Accuracy: {np.mean(metrics['accuracy']):.4f} ± {np.std(metrics['accuracy']):.4f}")
    print(f"Mean F1-Score: {np.mean(metrics['f1']):.4f} ± {np.std(metrics['f1']):.4f}")
    print(f"Mean Precision:{np.mean(metrics['precision']):.4f} ± {np.std(metrics['precision']):.4f}")
    print(f"Mean Recall:   {np.mean(metrics['recall']):.4f} ± {np.std(metrics['recall']):.4f}")

In [10]:
cv_pipeline_ensemble(X_ppmi, Y_ppmi)


=================== FOLD 1 ===================


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


ROC-AUC: 0.7778 | PR-AUC: 0.8975
Accuracy: 0.6800 | F1: 0.7647

=================== FOLD 2 ===================


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


ROC-AUC: 0.6508 | PR-AUC: 0.8667
Accuracy: 0.5600 | F1: 0.6667

=================== FOLD 3 ===================


...

ROC-AUC: 0.6555 | PR-AUC: 0.8450
Accuracy: 0.7083 | F1: 0.8000

=================== FOLD 9 ===================


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


ROC-AUC: 0.7647 | PR-AUC: 0.8998
Accuracy: 0.7083 | F1: 0.8000

=================== FOLD 10 ===================


/root/miniconda3/envs/rapids-26.04/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


ROC-AUC: 0.6639 | PR-AUC: 0.8408
Accuracy: 0.6667 | F1: 0.7500

================ FINAL ENSEMBLE RESULTS ================
Mean ROC-AUC:  0.6841 ± 0.0723
Mean PR-AUC:   0.8561 ± 0.0572
Mean Accuracy: 0.6820 ± 0.0630
Mean F1-Score: 0.7714 ± 0.0535
Mean Precision:0.7919 ± 0.0412
Mean Recall:   0.7556 ± 0.0804


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from cuml.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

variances = np.var(X_ppmi, axis=0)
threshold = np.percentile(variances, 75) 
high_var_mask = variances > threshold
X_high_var = X_ppmi[:, high_var_mask]


# 1. Scale 100% of the High-Variance Data
scaler_final = StandardScaler()
X_final_scaled = scaler_final.fit_transform(X_high_var)

if hasattr(X_final_scaled, 'get'):
    X_final_cpu = X_final_scaled.get()
    Y_final_cpu = Y_ppmi.get() if hasattr(Y_ppmi, 'get') else Y_ppmi
else:
    X_final_cpu = X_final_scaled
    Y_final_cpu = Y_ppmi

# 2. Select Top 500 Features on 100% of the Data
selector_final = SelectKBest(score_func=f_classif, k=500)
X_final_sel = selector_final.fit_transform(X_final_cpu, Y_final_cpu)

high_var_cols = X_pd_ppmi.columns[high_var_mask]
final_selected_features = high_var_cols[selector_final.get_support()]

# 3. Train Final Model 1: Logistic Regression
final_logreg = LogisticRegression(
    penalty='l2', C=0.1, class_weight='balanced', 
    solver='liblinear', max_iter=1000, random_state=42
)
final_logreg.fit(X_final_sel, Y_final_cpu)

# 4. Train Final Model 2: XGBoost
final_xgb = xgb.XGBClassifier(
    tree_method="hist", device="cuda", objective="binary:logistic",
    n_estimators=200, max_depth=2, learning_rate=0.05,
    colsample_bytree=0.3, subsample=0.8, random_state=42,
    scale_pos_weight=(Y_final_cpu == 0).sum() / (Y_final_cpu == 1).sum()
)
final_xgb.fit(X_final_sel, Y_final_cpu, verbose=False)

print("Ensemble successfully trained on 100% of data!")

# 5. Extract Feature Importances 
# We use the LogReg coefficients for a clean, directional interpretation 
# (Positive = Drives toward PD, Negative = Drives toward Control)
coefficients = final_logreg.coef_[0]

final_feature_df = pd.DataFrame({
    'CpG': final_selected_features,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
}).sort_values(by='Absolute_Importance', ascending=False)

print("\n--- Top 10 Ensemble-Backed Biomarkers ---")
print(final_feature_df.head(10))


In [ ]:
final_feature_df = final_feature_df.merge(df_beta_means[['delta_beta']], left_on='CpG', right_index=True, how='left')
final_feature_df.sort_values(by='Absolute_Importance', ascending=False).head(10)

In [ ]:
final_feature_df.to_csv('/root/data/ppmi/final_ensemble_pd_biomarkers.csv', index=False)

In [ ]:
%conda install bioconda::gseapy

In [ ]:
final_feature_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp

manifest_path = '/root/data/ppmi/infinium-epic-v1.csv' 

print("Loading annotation manifest...")
# We only load the columns we care about to save memory
manifest_epic = pd.read_csv(
    manifest_path, 
    skiprows=7,
    usecols=['Name', 'CHR', 'MAPINFO', 'UCSC_RefGene_Name', 'UCSC_RefGene_Group'],
    dtype=str,
    encoding='utf-8'
)

top_cpgs_epic = final_feature_df[['CpG']].head(500)

# Merge our top SHAP features with the genomic annotations
mapped_cpgs_epic = pd.merge(top_cpgs_epic, manifest_epic, left_on='CpG', right_on='Name', how='left')

# --- 3. Clean Up Illumina's Gene Names ---
# Illumina manifests often list multiple genes per probe separated by semicolons
# (e.g., "GENE1;GENE1;GENE2"). We need to split these and get a unique list.
print("Extracting unique gene targets...")

# Drop probes that don't map to a known gene
genes_raw = mapped_cpgs_epic['UCSC_RefGene_Name'].dropna()

# Split semicolons, flatten the list, and remove duplicates
gene_list = genes_raw.str.split(';').explode().unique().tolist()

# Remove any empty strings just in case
gene_list = [g for g in gene_list if g.strip() != '']

print(f"Mapped {len(top_cpgs_epic)} CpG sites to {len(gene_list)} unique genes.")

# --- 4. Functional Enrichment Analysis using GSEApy ---
print("Running Enrichr API (requires internet connection)...")

# Query lots of databases to gain the most enrichment information w. statistical relevance as possible
databases = ['GO_Biological_Process_2026', 'KEGG_2021_Human', 
'ENCODE_Histone_Modifications_2015',
'ENCODE_TF_ChIP-seq_2015',
'Elsevier_Pathway_Collection',
'GO_Cellular_Component_2026',
'GO_Molecular_Function_2026',
'Ligand_Perturbations_from_GEO_down',
'Ligand_Perturbations_from_GEO_up',
'Microbe_Perturbations_from_GEO_down',
'Microbe_Perturbations_from_GEO_up',
'Panther_2016',
'ProteomicsDB_2020',
'Reactome_Pathways_2024',
'RummaGEO_GenePerturbations_2025',
'SynGO_2024',
'TRANSFAC_and_JASPAR_PWMs',
'TRRUST_Transcription_Factors_2019',
'The_Kinase_Library_2024',
'WikiPathways_2024_Human']

enrichment_results = gp.enrichr(
    gene_list=gene_list,
    gene_sets=databases,
    organism='human',
    outdir=None # Set to a path if you want to save the raw tables automatically
)

# --- 5. Visualize the Results ---
# Get the results table
res_df = enrichment_results.results

# Filter for statistically significant pathways (Adjusted P-value < 0.05)
significant_pathways_ppmi = res_df[res_df['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')

if significant_pathways_ppmi.empty:
    print("\nNo pathways reached statistical significance (Adj. P < 0.05).")
else:
    print(f"\nFound {len(significant_pathways_ppmi)} significant pathways!")
    
    # Plot the top 15 pathways
    plt.figure(figsize=(10, 8))
    
    # We use -log10(P-value) for the x-axis (longer bar = more significant)
    top_plot = significant_pathways_ppmi.head(15).copy()
    top_plot['-log10(Adj. P)'] = -np.log10(top_plot['Adjusted P-value'])
    
    sns.barplot(
        data=top_plot,
        x='-log10(Adj. P)',
        y='Term',
        hue='Gene_set',
        dodge=False,
        palette='viridis'
    )
    
    plt.title('Top 15 Enriched Pathways (SHAP-selected Genes)', fontweight='bold')
    plt.xlabel('-log10(Adjusted P-value)')
    plt.ylabel('')
    plt.axvline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (P=0.05)')
    plt.legend(title='Database', loc='lower right')
    plt.tight_layout()
    plt.savefig('shap_pathway_enrichment.png', dpi=300)
    plt.show()

# Optional: Save the mapping and enrichment results to CSV for your records
mapped_cpgs_epic.to_csv('top_cpgs_annotated.csv', index=False)
significant_pathways_ppmi.to_csv('enrichment_results.csv', index=False)

In [ ]:
manifest_450k = pd.read_csv(
    '/root/data/gse/manifest-450.csv',
    usecols=['Name', 'CHR', 'MAPINFO', 'UCSC_RefGene_Name', 'UCSC_RefGene_Group'],
    dtype=str,
    encoding='utf-8',
    skiprows=7
)

In [ ]:
epic_cpgs = set(final_feature_df['CpG'])
i450k_cpgs = set(manifest_450k['Name'])
shared_cpgs = epic_cpgs.intersection(i450k_cpgs)
epic_only_cpgs = epic_cpgs.difference(i450k_cpgs)

In [ ]:
print(f"Total EPIC: {len(epic_cpgs)}")
print(f"Present in BOTH arrays (450k Compatible): {len(shared_cpgs)}")
print(f"Unique to EPIC (850k/v2) array: {len(epic_only_cpgs)}")

**Check what Δβ looks like for the shared CpG's**

In [ ]:
final_feature_df[final_feature_df['CpG'].isin(shared_cpgs)].sort_values(by='delta_beta', ascending=False).head()

In [ ]:
X_pd_ppmi_subset = X_pd_ppmi[X_pd_ppmi.columns.intersection(shared_cpgs)]
X_pd_ppmi_subset_np = X_pd_ppmi_subset.to_numpy(dtype="float32")

X_train, X_test, Y_train, Y_test = train_test_split(X_pd_ppmi_subset_np, Y_ppmi, test_size=0.2, random_state=42)

final_scaler = StandardScaler()

X_train_scaled = final_scaler.fit_transform(X_train)
X_test_scaled = final_scaler.transform(X_test)

model = LogisticRegression(
    penalty='l2', C=0.1, class_weight='balanced', 
    solver='liblinear', max_iter=1000, random_state=42
)
model.fit(X_train_scaled, Y_train)

In [ ]:
pred = model.predict_proba(X_test_scaled)[:, 1]

auc_roc = roc_auc_score(Y_test, pred)
auc_pr = average_precision_score(Y_test, pred)

print(f"ROC-AUC on 450k-compatible features: {auc_roc:.4f}")
print(f"PR-AUC on 450k-compatible features: {auc_pr:.4f}")

In [ ]:
pred = model.predict(X_test_scaled)

auc_roc = roc_auc_score(Y_test, pred)
auc_pr = average_precision_score(Y_test, pred)

print(f"ROC-AUC on 450k-compatible features: {auc_roc:.4f} ")
print(f"PR-AUC on 450k-compatible features: {auc_pr:.4f}")

In [ ]:
gse_src_data_path = "/root/data/gse/data_gse111629.parquet"
df_gse = pd.read_parquet(gse_src_data_path)

In [ ]:
import pandas as pd
import numpy as np
from cuml.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

print("--- Preparing External Validation (159 Probes) ---")

# 1. Identify the missing probes
shared_cpgs_list = list(shared_cpgs) # From your previous intersection
gse_cpgs = set(df_gse.columns)
missing_in_gse = set(shared_cpgs_list) - gse_cpgs

print(f"Features missing in GSE: {missing_in_gse}")

# =========================================================
# PHASE A: Train the 159-Feature Model on ORIGINAL Data
# =========================================================
print("\nTraining subset model on original 159 features...")

# Extract ONLY the 159 features from your original dataset (X_pd)
X_train_159 = X_pd_ppmi[shared_cpgs_list].copy()
Y_train_all = Y_ppmi # Assuming Y_ppmi is still in memory from your pipeline

# Create a dedicated scaler for these 159 features
scaler_159 = StandardScaler()
X_train_159_scaled = scaler_159.fit_transform(X_train_159)

def to_cpu_numpy(data):
    type_str = str(type(data)).lower()
    if 'cupy' in type_str:
        return data.get()
    elif 'cudf' in type_str:
        return data.to_numpy()
    else:
        return np.array(data) # Handles Pandas and standard NumPy

X_train_159_cpu = to_cpu_numpy(X_train_159_scaled)
Y_train_cpu = to_cpu_numpy(Y_train_all)

# Train the subset Logistic Regression
val_logreg = LogisticRegression(
    penalty='l2', C=0.1, class_weight='balanced', 
    solver='liblinear', max_iter=1000, random_state=42
)
val_logreg.fit(X_train_159_cpu, Y_train_cpu)

# =========================================================
# PHASE B: Prepare and Impute the GSE Data
# =========================================================
print("\nPreparing and imputing GSE dataset...")

# Create a copy of the available features in GSE
available_cpgs = [cpg for cpg in shared_cpgs_list if cpg not in missing_in_gse]
X_gse_159 = df_gse[available_cpgs].copy()

# Impute the missing features using the ORIGINAL training data mean
for missing_probe in missing_in_gse:
    train_mean = X_train_159[missing_probe].mean()
    X_gse_159[missing_probe] = train_mean
    print(f"Imputed '{missing_probe}' with original mean value: {train_mean:.4f}")

# ENFORCE EXACT COLUMN ORDER (Critical for the scaler and model)
X_gse_159 = X_gse_159[shared_cpgs_list]

# =========================================================
# PHASE C: Scale and Predict
# =========================================================
print("\nScaling GSE data and generating predictions...")

# Scale GSE data using the original parameters
X_gse_scaled = scaler_159.transform(X_gse_159)
X_gse_scaled_cpu = to_cpu_numpy(X_gse_scaled)

# Extract the Target labels from GSE (Ensure they are mapped 0=Control, 1=PD)
Y_gse = df_gse['Sample_Group'] # UPDATE THIS to your actual GSE label column name
Y_gse = Y_gse.map({"Control": 0, "PD": 1}).to_numpy(dtype="float32")

# Generate Probabilities
gse_probs = val_logreg.predict_proba(X_gse_scaled_cpu)[:, 1]

# Calculate Metrics
gse_auc = roc_auc_score(Y_gse, gse_probs)
gse_pr = average_precision_score(Y_gse, gse_probs)

print("\n================ GSE VALIDATION RESULTS ================")
print(f"External Validation ROC-AUC: {gse_auc:.4f}")
print(f"External Validation PR-AUC:  {gse_pr:.4f}")

In [ ]:
from cuml.preprocessing import StandardScaler

print("\n--- PHASE C: Independent Scaling (Batch Effect Bypass) ---")

# We create a BRAND NEW scaler specifically to center the GSE dataset 
# on its own internal lab baseline
scaler_gse_independent = StandardScaler()

# Notice we are now using fit_transform ON the GSE data
X_gse_scaled_independent = scaler_gse_independent.fit_transform(X_gse_159)

# Convert to CPU numpy array
def to_cpu_numpy(data):
    type_str = str(type(data)).lower()
    if 'cupy' in type_str: return data.get()
    elif 'cudf' in type_str: return data.to_numpy()
    else: return np.array(data)

X_gse_scaled_cpu = to_cpu_numpy(X_gse_scaled_independent)

# Generate Probabilities using your PRE-TRAINED model
gse_probs_independent = val_logreg.predict_proba(X_gse_scaled_cpu)[:, 1]

# Calculate Metrics
gse_auc_ind = roc_auc_score(Y_gse, gse_probs_independent)
gse_pr_ind = average_precision_score(Y_gse, gse_probs_independent)

print("\n================ GSE INDEPENDENT VALIDATION ================")
print(f"Batch-Bypassed ROC-AUC: {gse_auc_ind:.4f}")
print(f"Batch-Bypassed PR-AUC:  {gse_pr_ind:.4f}")